# Vine copula example

Fit a truncated 6-dimensional R-vine on crypto returns, compare it with elliptical copulas, then sample and predict from the fitted vine.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from pyscarcopula._utils import pobs
from pyscarcopula import GaussianCopula, RVineCopula, StudentCopula
from pyscarcopula.stattests import gof_test


DATA_DIR = Path('data')
if not DATA_DIR.exists():
    DATA_DIR = Path('..') / 'data'

## Data

In [2]:
tickers = ['BTC-USD', 'ETH-USD', 'BNB-USD', 'ADA-USD', 'XRP-USD', 'DOGE-USD']
crypto_prices = pd.read_csv(DATA_DIR / 'crypto_prices.csv', index_col=0, sep=';')

returns = np.log(crypto_prices[tickers] / crypto_prices[tickers].shift(1))[1:251].values
u = pobs(returns)

u.shape

(250, 6)

## Fit R-vine

In [3]:
vine = RVineCopula()
vine.fit(u, method='scar-tm-ou', truncation_level=2)

print(f'logL: {vine.log_likelihood():.2f}')
print(f'n_parameters: {vine.n_parameters}')
print(f'AIC: {vine.aic:.2f}')
print(f'BIC: {vine.bic:.2f}')
print('Variable order:', [int(vine.matrix[vine.d - 1 - col, col]) for col in range(vine.d)])

logL: 885.75
n_parameters: 27
AIC: -1717.51
BIC: -1622.43
Variable order: [4, 3, 2, 1, 0, 5]


## Model comparison

In [4]:
vine_mle = RVineCopula()
vine_mle.fit(u, method='mle', truncation_level=2)

student = StudentCopula()
student.fit(u)

gaussian = GaussianCopula()
gaussian.fit(u)

models = {
    'R-vine SCAR-TM': vine,
    'R-vine MLE': vine_mle,
    'Student-t': student,
    'Gaussian': gaussian,
}

print(f"{'Model':<16} {'logL':>10} {'GoF p-value':>12}")
print('-' * 40)
for name, model in models.items():
    if isinstance(model, RVineCopula):
        log_likelihood = model.log_likelihood()
    else:
        log_likelihood = model.log_likelihood(u)

    gof = gof_test(model, u, to_pobs=False)
    print(f'{name:<16} {log_likelihood:>10.2f} {gof.pvalue:>12.4f}')

Model                  logL  GoF p-value
----------------------------------------
R-vine SCAR-TM       885.75       0.9413
R-vine MLE           836.96       0.0639
Student-t            819.75       0.4753
Gaussian             761.00       0.0000


## Sampling and prediction

In [5]:
sample_u = vine.sample(2000, rng=np.random.default_rng(2024))
predicted_u = vine.predict(5000, u=u, horizon='next', rng=np.random.default_rng(2025))

print('Sample shape:', sample_u.shape)
print('Prediction shape:', predicted_u.shape)
print('Sample GoF p-value:', round(gof_test(vine, sample_u, to_pobs=False).pvalue, 4))
print(pd.Series(predicted_u.mean(axis=0), index=tickers, name='predicted_mean'))

Sample shape: (2000, 6)
Prediction shape: (5000, 6)
Sample GoF p-value: 0.4532
BTC-USD     0.500461
ETH-USD     0.502038
BNB-USD     0.502812
ADA-USD     0.506168
XRP-USD     0.503698
DOGE-USD    0.502620
Name: predicted_mean, dtype: float64


## Conditional prediction

In [6]:
variable_order = [int(vine.matrix[vine.d - 1 - col, col]) for col in range(vine.d)]
given = {variable_order[-1]: 0.4}

conditional_u = vine.predict(
    3000,
    u=u,
    given=given,
    horizon='current',
    rng=np.random.default_rng(2026),
)

print('Condition:', {tickers[idx]: value for idx, value in given.items()})
print('Conditional shape:', conditional_u.shape)
print('Fixed coordinate:', np.unique(conditional_u[:, next(iter(given))]))
print(pd.Series(conditional_u.mean(axis=0), index=tickers, name='conditional_mean'))

Condition: {'DOGE-USD': 0.4}
Conditional shape: (3000, 6)
Fixed coordinate: [0.4]
BTC-USD     0.431078
ETH-USD     0.460767
BNB-USD     0.486766
ADA-USD     0.471232
XRP-USD     0.475962
DOGE-USD    0.400000
Name: conditional_mean, dtype: float64


## RNG reproducibility

In [7]:
seed = 4040
u_a = vine.predict(1000, u=u, given=given, horizon='current', rng=np.random.default_rng(seed))
u_b = vine.predict(1000, u=u, given=given, horizon='current', rng=np.random.default_rng(seed))
u_c = vine.predict(1000, u=u, given=given, horizon='current', rng=np.random.default_rng(seed + 1))

print('Same seed -> identical:', np.allclose(u_a, u_b))
print('Different seed -> identical:', np.allclose(u_a, u_c))

Same seed -> identical: True
Different seed -> identical: False
